# Build Supervised Multi-Agent AI Systems with Manager-Worker Pattern

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/multi-agent-workflows/tutorial_manager_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Learn to build a hierarchical manager-worker system where a manager agent delegates tasks, reviews outputs, and ensures quality through active supervision.

**Pattern:** Manager plans → Delegates → Reviews → Requests revisions → Synthesizes

```
Project: "Build a calculator with add, subtract, multiply"
  ↓
Manager plans:
  Task 0: code - implement add function
  Task 1: code - implement subtract function  
  Task 2: code - implement multiply function
  Task 3: code - combine into calculator (depends: 0,1,2)
  ↓
Execute Task 0:
  Worker: "def add(a, b): return a+b"
  Manager: "Quality 6/10 - missing docstring" → NEEDS REVISION
  Worker: "def add(a, b): """Add two numbers""" return a+b"
  Manager: "Quality 9/10" → APPROVED ✅
  ↓
[Repeat for tasks 1, 2, 3]
  ↓
Manager synthesizes final deliverable
```

**Key concepts:** Hierarchical coordination, quality gates, revision cycles, active supervision, dependency management.

**Why Manager-Worker?**
- ✅ **Quality control**: Manager reviews ensure standards met
- ✅ **Adaptive**: Manager adjusts based on worker performance
- ✅ **Accountability**: Clear hierarchy and approval process
- ✅ **Iterative improvement**: Workers revise based on feedback
- ✅ **Coordination**: Manager handles dependencies and integration

---

## Setup

Project structure: `agents/`, `tools/`, `workflows/`, `utils/`

**Configuration:** Shared Flyte environment with Docker image + secrets. Agents inherit this config but can override for custom resources.

In [1]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/multi-agent-workflows/
    !uv pip install -r requirements.txt
    
# this is just for viewing the code files within the notebook
from utils.file_viewer import view_file

In [2]:
view_file("requirements.txt")

In [3]:
view_file("config.py")

## Connect to Flyte Cluster
You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.

Flyte gives you...

- If you don't have a Flyte cluster you can request demo access by filling out the form [here](https://flyte.org/).
- If you already have a Flyte cluster, you can connect to it by setting your endpoint in the Flyte configuration.


In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --auth-type headless\
    --builder remote \
    --domain development \
    --project flytesnacks

You can now adjust the configuration by modifying the `.flyte/config.yaml` file.

In [4]:
view_file(".flyte/config.yaml")

## Set your API Key(s)

The project is setup to read in secrets from a `.env` file.

You can create this file in the root of this tutorial `tutorials/multi-agent-workflows` and add your API keys there.

But if you prefer to just enter a key once in this notebook you can run the cell below:

In [ ]:
# Skip if API key is already set in .env or environment
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

To run on the remote Flyte cluster, add the API keys as secrets. 

You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.


In [ ]:
# run this and enter your API key as the input
!flyte create secret OPENAI_API_KEY

## Run the Agent

At this point you should be setup to run the manager-worker workflow. 

I suggest giving it a try before we walk through the code in the next section.

**Run locally:**

In [1]:
!python -m workflows.manager --local --request "Calc|ulate 5 factorial and explain the result" --quality-threshold 7 --max-revisions 2

Running workflow LOCALLY with flyte.init()

=== Manager-Worker Multi-Agent Workflow ===
Project: Calc|ulate 5 factorial and explain the result
Quality threshold: 7/10
Max revisions per task: 2

MANAGER-WORKER WORKFLOW
Project: Calc|ulate 5 factorial and explain the result
Quality threshold: 7/10

PHASE 1 - MANAGER PLANNING

[Manager] Analyzing project and creating delegation plan...

[Manager] Created plan with 3 task(s):
  Task 0: math - Calculate the factorial of 5.... (no dependencies)
  Task 1: string - Explain what a factorial is in simple terms.... (no dependencies)
  Task 2: string - Explain the result of 5 factorial in a clear and concise man... (depends on: [0, 1])

PHASE 2 - SUPERVISED EXECUTION

[Manager] Delegating Task 0 to math worker...
[Manager] Task: Calculate the factorial of 5.
[Math Agent] Processing: Calculate the factorial of 5.
[Math Tool] Multiplying 5.0 * 4.0
[Math Tool] Multiplying 20.0 * 3.0
[Math Tool] Multiplying 60.0 * 2.0
[Math Tool] Multiplying 120.0 * 1

**Run on the remote Flyte cluster:**

The first time running the agent a container image will be built and pushed to the Flyte cluster.

This may take some time depending on the size of your dependencies.

In [3]:
!python -m workflows.manager --request "Calculate 5 factorial and explain the result" --quality-threshold 7 --max-revisions 2

Running workflow REMOTELY with flyte.init_from_config()

=== Manager-Worker Multi-Agent Workflow ===
Project: Calculate 5 factorial and explain the result
Quality threshold: 7/10
Max revisions per task: 2

16:59:01.639258 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8 found. Skip    
                         building.                                              
16:59:01.640663 WARNING  _deploy.py:376 -  Built Image for environment base_env,
                         image:                                                 
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8                

Execution: rftj7nx25lwnkhdn8lk4
URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/flytesnacks/runs/rftj7nx25lwnkhdn8lk4



# Code Walkthrough

Let's walk through the code to understand how the planner agent is structured and how it works.

We'll cover all the key file types, but you can explore all the agents and tools in their respective folders.

## Infrastructure

**Decorators** - Registration system for agents and tools. 

This allows for easy addition and management of new agents and tools within the workflow.

In [ ]:
view_file("agents/planner_agent.py")

---

## Orchestrator - The Agentic Workflow

Executes plans with dependency-aware parallelism.

**Flow:**
1. Get plan from planner
2. Loop: Find steps with satisfied dependencies → Execute in parallel → Mark complete
3. For dependent steps: Inject previous results via `build_task_with_context()`

**Context passing:**
```python
# Step 2 depends on steps 0 and 1
task = """
RESULTS FROM PREVIOUS STEPS:
  - Step 0 (math): 5
  - Step 1 (math): 11

YOUR TASK:
Add the results
"""
```

**Key features:** Automatic parallelization (`asyncio.gather`), result propagation, circular dependency detection.

In [ ]:

view_file("workflows/planner.py")

## Manager-Worker Orchestrator - Hierarchical Supervision

The manager-worker pattern uses hierarchical coordination with active quality control.

**Three-Phase Flow:**

### Phase 1: Manager Planning
Manager analyzes project and creates delegation plan:
```python
delegation_plan = [
    WorkerTask(task_id=0, agent="code", description="...", dependencies=[]),
    WorkerTask(task_id=1, agent="string", description="...", dependencies=[0])
]
```

### Phase 2: Supervised Execution
For each task (respecting dependencies):
1. **Delegate**: Manager assigns task to worker
2. **Execute**: Worker completes task
3. **Review**: Manager evaluates output
   ```python
   review = {
       "quality_score": 6,  # 1-10 scale
       "issues": ["Missing docstring", "No error handling"],
       "approved": False,
       "feedback": "Add docstrings and handle edge cases"
   }
   ```
4. **Iterate**: If score < threshold, worker revises based on feedback
5. **Approve**: Once quality met or max revisions reached

### Phase 3: Final Synthesis
Manager integrates all worker outputs into coherent final deliverable.

**Quality Gate System:**
```python
# Worker submits output
output = worker.execute(task)

# Manager reviews
for revision in range(max_revisions):
    review = manager.review(output)
    
    if review.quality_score >= threshold:
        manager.approve()  # ✅ Quality gate passed
        break
    else:
        output = worker.revise(review.feedback)  # 🔄 Revision cycle
```

**Key differences from other patterns:**

| Feature | Manager-Worker | Planner | Debate |
|---------|----------------|---------|--------|
| **Hierarchy** | Manager → Workers | Flat orchestration | Peer collaboration |
| **Supervision** | Active reviews | Fire-and-forget | Self-organized |
| **Quality control** | Manager approves | No reviews | Mutual critique |
| **Revisions** | Manager requests | No revisions | Self-refinement |
| **Coordination** | Manager handles | Automatic via plan | Emergent consensus |

**Agent routing:** Uses `agent_registry` for dynamic dispatch (same as all workflows).

In [5]:
view_file("workflows/manager.py")

---

## Running the Workflow

**Local (development):**
```bash
python -m workflows.manager --local \
  --request "your project" \
  --quality-threshold 7 \
  --max-revisions 2
```
In-process execution, fast iteration.

**Remote (production):**
```bash
python -m workflows.manager \
  --request "your project" \
  --quality-threshold 8 \
  --max-revisions 3
```
Distributed Flyte cluster, scalable and observable.

**Parameters:**
- `--quality-threshold`: Minimum score (1-10) to approve. Higher = stricter. Default: 7
- `--max-revisions`: Max revision cycles per task. Default: 2

**Try these:**

In [ ]:
# Multi-step calculation project
!python -m workflows.manager --local --request "Calculate 10 factorial and explain what factorials are" --quality-threshold 7 --max-revisions 2

In [ ]:
# Code generation with high quality standards
!python -m workflows.manager --local --request "Write Python functions for calculating area of circle, square, and triangle" --quality-threshold 8 --max-revisions 3

In [ ]:
# Research and analysis project
!python -m workflows.manager --local --request "Search for Python programming info and create a summary" --quality-threshold 7 --max-revisions 2

---

## Key Takeaways

**Manager-Worker Pattern:**
- **Hierarchical**: Clear manager → worker authority structure
- **Quality-focused**: Manager reviews ensure standards met
- **Iterative**: Workers revise based on manager feedback
- **Coordinated**: Manager handles dependencies and integration
- **Accountable**: Explicit approval process with quality scores

**When to use Manager-Worker:**

| Use Manager-Worker When | Use Other Pattern When |
|-------------------------|------------------------|
| Quality standards critical | Speed is priority |
| Need oversight/approval | Trust autonomous agents |
| Complex integration required | Simple independent tasks |
| Want revision cycles | One-shot execution sufficient |
| Hierarchical structure natural | Flat/peer collaboration better |

**Complete Pattern Comparison:**

| Pattern | Coordination | Best For |
|---------|--------------|----------|
| **Planner** | Static plan → parallel waves | Known decomposition, maximize speed |
| **ReAct** | Adaptive single agent | Exploration, unknown steps |
| **Reflection** | Self-improvement | High quality single outputs |
| **Sequential** | Fixed pipeline | Predictable workflows |
| **Debate** | Peer collaboration | Accuracy through consensus |
| **Manager-Worker** | Hierarchical supervision | Quality control, complex projects |

**Real-world applications:**
- **Software development**: Manager oversees design, implementation, testing, deployment
- **Content production**: Editor reviews writer submissions
- **Research projects**: PI coordinates researcher tasks
- **Consulting**: Engagement manager oversees specialist consultants
- **Quality assurance**: QA lead reviews tester outputs

**Tuning quality vs speed:**
```python
# Strict quality, more time
--quality-threshold 9 --max-revisions 5

# Balanced
--quality-threshold 7 --max-revisions 2

# Fast, lenient
--quality-threshold 5 --max-revisions 1
```

**Architecture benefits:**
- Same agents/tools work with all patterns
- Type-safe, observable, scalable with Flyte
- Combine patterns (e.g., manager delegates to debate teams)

**Next steps:**
1. Experiment with different quality thresholds
2. Try complex multi-step projects
3. Compare manager vs planner for same project
4. Combine patterns (manager coordinates multiple workflow types)

---

## Resources

- Full code: `tutorials/multi-agent-workflows/`
- Planner tutorial: [tutorial_planner_agent.ipynb](tutorial_planner_agent.ipynb)
- ReAct tutorial: [tutorial_react_agent.ipynb](tutorial_react_agent.ipynb)
- Reflection tutorial: [tutorial_reflection_agent.ipynb](tutorial_reflection_agent.ipynb)
- Sequential tutorial: [tutorial_sequential_agent.ipynb](tutorial_sequential_agent.ipynb)
- Debate tutorial: [tutorial_debate_agent.ipynb](tutorial_debate_agent.ipynb)
- Flyte docs: https://docs.flyte.org
- Questions? Join the Flyte community Slack!